# 02 — Data Preprocessing & Cleansing
**Phishing URL Detection — CS 4200 Final Project**  
**Author:** Joe Casperson

**Chapter Coverage:** Chapter 9 — Data Preprocessing and Cleansing

**Purpose:**  
Audit the raw data loaded in notebook 01, apply a documented cleaning pipeline,  
and produce a verified clean dataset ready for visualization and modeling.

**Cleaning Steps:**  
1. Load raw data from PostgreSQL  
2. Pre-clean audit (Python + SQL)  
3. Drop sparse columns (>40% null)  
4. Remove duplicate rows  
5. Impute remaining null values  
6. Verify data types and value ranges  
7. Post-clean audit and before/after comparison  
8. Export to PostgreSQL `clean_urls` + `data/clean/clean_urls.csv`  

**Each operation is logged to MongoDB** for a full reproducible audit trail.

---

## 1. Imports and Connections

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pymongo import MongoClient
from dotenv import load_dotenv
import datetime
import os

load_dotenv()

# ── PostgreSQL ────────────────────────────────────────────────────────────────
conn_str = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}/{os.getenv('POSTGRES_DB')}"
)
engine = create_engine(conn_str)

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
    print('✓ PostgreSQL connected')

# ── MongoDB ───────────────────────────────────────────────────────────────────
mongo_client = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017/'))
prep_log     = mongo_client.phishdb.preprocessing_log
print('✓ MongoDB connected')

# ── Helper: log every cleaning operation to MongoDB ───────────────────────────
def log_operation(operation, rows_before, rows_after, detail=''):
    """Insert one document per cleaning step into preprocessing_log."""
    prep_log.insert_one({
        'operation'    : operation,
        'rows_before'  : int(rows_before),
        'rows_after'   : int(rows_after),
        'rows_affected': int(rows_before - rows_after),
        'detail'       : detail,
        'timestamp'    : datetime.datetime.utcnow()
    })
    print(f'  [{operation}] {rows_before:,} → {rows_after:,} rows '
          f'({rows_before - rows_after:,} affected)')

print('✓ MongoDB log helper ready')

✓ PostgreSQL connected
✓ MongoDB connected
✓ MongoDB log helper ready


## 2. Load Raw Data from PostgreSQL

In [2]:
df = pd.read_sql('SELECT * FROM urls ORDER BY url_id', engine)

# Drop the auto-generated DB columns — not needed for analysis
df = df.drop(columns=['url_id', 'inserted_at'], errors='ignore')

print(f'✓ Loaded {len(df):,} rows from PostgreSQL urls table')
print(f'  Columns: {len(df.columns)}')

✓ Loaded 15,367 rows from PostgreSQL urls table
  Columns: 80


## 3. Pre-Clean Audit

Establish the baseline state before any modifications.  

In [3]:
# ── Shape and dtypes ──────────────────────────────────────────────────────────
print('── Pre-Clean Audit ─────────────────────────────')
print(f'  Shape      : {df.shape}')
print(f'  Duplicates : {df.duplicated().sum():,}')
print(f'  Total nulls: {df.isnull().sum().sum():,}')

── Pre-Clean Audit ─────────────────────────────
  Shape      : (15367, 80)
  Duplicates : 544
  Total nulls: 9,669


In [4]:
# ── Null counts per column ────────────────────────────────────────────────────
null_counts  = df.isnull().sum()
null_pct     = (null_counts / len(df) * 100).round(2)
null_report  = pd.DataFrame({
    'null_count': null_counts,
    'null_pct'  : null_pct
}).query('null_count > 0').sort_values('null_pct', ascending=False)

print('── Null Values by Column (Before) ──────────────')
print(null_report.to_string())

── Null Values by Column (Before) ──────────────
                          null_count  null_pct
numberrate_extension            7355     47.86
entropy_directoryname           1826     11.88
avgpathtokenlen                  271      1.76
entropy_filename                 190      1.24
numberrate_directoryname           9      0.06
numberrate_filename                9      0.06
numberrate_afterpath               3      0.02
entropy_extension                  3      0.02
entropy_afterpath                  3      0.02


In [5]:
# ── Class balance before cleaning ─────────────────────────────────────────────
label_before = df['label'].value_counts().sort_index()
print('── Class Balance (Before) ──────────────────────')
print(f'  Benign   (0): {label_before[0]:,} ({label_before[0]/len(df)*100:.2f}%)')
print(f'  Phishing (1): {label_before[1]:,} ({label_before[1]/len(df)*100:.2f}%)')

── Class Balance (Before) ──────────────────────
  Benign   (0): 7,781 (50.63%)
  Phishing (1): 7,586 (49.37%)


In [6]:
# ── Key feature statistics before cleaning ────────────────────────────────────
key_cols = ['label', 'urllen', 'domain_token_count', 'entropy_url',
            'symbolcount_url', 'numberrate_url', 'numberofdotsinurl']
print('── Key Feature Statistics (Before) ─────────────')
df[key_cols].describe().round(4)

── Key Feature Statistics (Before) ─────────────


,label,urllen,domain_token_count,entropy_url,symbolcount_url,numberrate_url,numberofdotsinurl
count,15367.0000,15367.0000,15367.0000,15367.0000,15367.0000,15367.0000,15367.000
mean,0.4937,76.0354,2.5437,0.7217,7.1851,0.0861,2.362
std,0.5000,33.4490,0.9449,0.0492,3.5428,0.1005,1.781
min,0.0000,37.0000,2.0000,0.4196,2.0000,0.0000,1.000
25%,0.0000,54.0000,2.0000,0.6872,5.0000,0.0000,1.000
50%,0.0000,68.0000,2.0000,0.7232,6.0000,0.0566,2.000
75%,1.0000,90.0000,3.0000,0.7579,8.0000,0.1250,3.000
max,1.0000,348.0000,19.0000,0.8697,47.0000,0.7619,20.000


## 4. SQL Audit Cross-Check

Run the Section 1 queries from `db/queries.sql` directly in Python  
to cross-verify the Python audit above. Both tools should agree.

In [7]:
print('── SQL Audit: Null Counts ───────────────────────')
sql_nulls = '''
    SELECT
        COUNT(*) FILTER (WHERE urllen              IS NULL) AS null_urllen,
        COUNT(*) FILTER (WHERE entropy_url         IS NULL) AS null_entropy_url,
        COUNT(*) FILTER (WHERE numberrate_extension IS NULL) AS null_numberrate_ext,
        COUNT(*) FILTER (WHERE entropy_directoryname IS NULL) AS null_entropy_dir,
        COUNT(*) FILTER (WHERE avgpathtokenlen      IS NULL) AS null_avgpathtokenlen
    FROM urls
'''
display(pd.read_sql(sql_nulls, engine))

── SQL Audit: Null Counts ───────────────────────


,null_urllen,null_entropy_url,null_numberrate_ext,null_entropy_dir,null_avgpathtokenlen
0,0,0,7355,1826,271


In [8]:
print('── SQL Audit: Class Distribution ───────────────')
sql_balance = '''
    SELECT
        label,
        CASE WHEN label = 1 THEN 'Phishing' ELSE 'Benign' END AS label_name,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM urls
    GROUP BY label
    ORDER BY label DESC
'''
display(pd.read_sql(sql_balance, engine))

── SQL Audit: Class Distribution ───────────────


,label,label_name,count,pct
0,1,Phishing,7586,49.37
1,0,Benign,7781,50.63


## 5. Cleaning Pipeline

Every operation is logged to MongoDB with before/after row counts.

In [9]:
print('── Cleaning Pipeline ───────────────────────────')
df_clean = df.copy()
starting_rows = len(df_clean)

── Cleaning Pipeline ───────────────────────────


In [10]:
# ── STEP 1: Drop sparse columns (>40% null) ───────────────────────────────────
# numberrate_extension is 47.86% null — too sparse to impute reliably.
# Dropping it is safer than imputing nearly half the column with medians.
SPARSE_THRESHOLD = 0.40
sparse_cols = null_pct[null_pct > SPARSE_THRESHOLD * 100].index.tolist()

before = len(df_clean)
df_clean = df_clean.drop(columns=sparse_cols, errors='ignore')

print(f'STEP 1 — Drop sparse columns (>{SPARSE_THRESHOLD*100:.0f}% null)')
print(f'  Columns dropped: {sparse_cols}')
print(f'  Columns remaining: {len(df_clean.columns)}')

log_operation(
    operation   = 'drop_sparse_columns',
    rows_before = before,
    rows_after  = len(df_clean),
    detail      = f'Dropped columns with >{SPARSE_THRESHOLD*100:.0f}% nulls: {sparse_cols}'
)

STEP 1 — Drop sparse columns (>40% null)
  Columns dropped: ['numberrate_extension']
  Columns remaining: 79
  [drop_sparse_columns] 15,367 → 15,367 rows (0 affected)


/tmp/ipykernel_158827/297134292.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'    : datetime.datetime.utcnow()


In [11]:
# ── STEP 2: Remove duplicate rows ────────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean.drop_duplicates()

print(f'STEP 2 — Remove duplicate rows')
log_operation(
    operation   = 'remove_duplicates',
    rows_before = before,
    rows_after  = len(df_clean),
    detail      = f'Removed exact duplicate rows'
)

STEP 2 — Remove duplicate rows
  [remove_duplicates] 15,367 → 14,823 rows (544 affected)


/tmp/ipykernel_158827/297134292.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'    : datetime.datetime.utcnow()


In [12]:
# ── STEP 3: Median imputation for remaining null columns ──────────────────────
# Remaining null columns have <12% missing — median imputation is appropriate.
# Median is preferred over mean for skewed distributions (URL features often are).
remaining_nulls = df_clean.isnull().sum()
cols_to_impute  = remaining_nulls[remaining_nulls > 0].index.tolist()

before = len(df_clean)
imputed_values = {}

for col in cols_to_impute:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    imputed_values[col] = round(float(median_val), 4)

print(f'STEP 3 — Median imputation for {len(cols_to_impute)} columns')
for col, val in imputed_values.items():
    print(f'  {col}: median = {val}')

log_operation(
    operation   = 'median_imputation',
    rows_before = before,
    rows_after  = len(df_clean),
    detail      = f'Imputed nulls with column medians: {imputed_values}'
)

STEP 3 — Median imputation for 8 columns
  avgpathtokenlen: median = 4.5
  numberrate_directoryname: median = 0.0
  numberrate_filename: median = 0.0
  numberrate_afterpath: median = -1.0
  entropy_directoryname: median = 0.7852
  entropy_filename: median = 0.8124
  entropy_extension: median = 0.0
  entropy_afterpath: median = -1.0
  [median_imputation] 14,823 → 14,823 rows (0 affected)


/tmp/ipykernel_158827/297134292.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'    : datetime.datetime.utcnow()


In [13]:
# ── STEP 4: Verify all numeric dtypes ────────────────────────────────────────
# All feature columns should be numeric. Log any that are not.
before      = len(df_clean)
non_numeric = df_clean.drop(columns=['label']).select_dtypes(exclude='number').columns.tolist()

print(f'STEP 4 — Dtype verification')
if non_numeric:
    print(f'  Non-numeric columns found: {non_numeric}')
    for col in non_numeric:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    print(f'  Converted to numeric')
else:
    print(f'  ✓ All feature columns are numeric')

log_operation(
    operation   = 'dtype_verification',
    rows_before = before,
    rows_after  = len(df_clean),
    detail      = f'Non-numeric columns found and converted: {non_numeric}' if non_numeric
                  else 'All columns confirmed numeric — no changes needed'
)

STEP 4 — Dtype verification
  ✓ All feature columns are numeric
  [dtype_verification] 14,823 → 14,823 rows (0 affected)


/tmp/ipykernel_158827/297134292.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'    : datetime.datetime.utcnow()


In [14]:
# ── STEP 5: Verify label integrity ───────────────────────────────────────────
# Confirm only valid label values (0 and 1) remain after cleaning
before       = len(df_clean)
invalid_labels = df_clean[~df_clean['label'].isin([0, 1])]

print(f'STEP 5 — Label integrity check')
if len(invalid_labels) > 0:
    print(f'  Invalid labels found: {len(invalid_labels)} rows — dropping')
    df_clean = df_clean[df_clean['label'].isin([0, 1])]
else:
    print(f'  ✓ All labels are valid (0 or 1)')

log_operation(
    operation   = 'label_integrity_check',
    rows_before = before,
    rows_after  = len(df_clean),
    detail      = f'Verified all labels are 0 or 1'
)

STEP 5 — Label integrity check
  ✓ All labels are valid (0 or 1)
  [label_integrity_check] 14,823 → 14,823 rows (0 affected)


/tmp/ipykernel_158827/297134292.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'    : datetime.datetime.utcnow()


## 6. Post-Clean Audit

Verify the cleaning pipeline worked correctly.  


In [15]:
print('── Post-Clean Audit ────────────────────────────')
print(f'  Shape      : {df_clean.shape}')
print(f'  Duplicates : {df_clean.duplicated().sum()}')
print(f'  Total nulls: {df_clean.isnull().sum().sum()}')

label_after = df_clean['label'].value_counts().sort_index()
print(f'  Benign   (0): {label_after[0]:,} ({label_after[0]/len(df_clean)*100:.2f}%)')
print(f'  Phishing (1): {label_after[1]:,} ({label_after[1]/len(df_clean)*100:.2f}%)')

── Post-Clean Audit ────────────────────────────
  Shape      : (14823, 79)
  Duplicates : 0
  Total nulls: 0
  Benign   (0): 7,464 (50.35%)
  Phishing (1): 7,359 (49.65%)


In [17]:
# ── Before / After comparison table ──────────────────────────────────────────
comparison = pd.DataFrame({
    'Metric': [
        'Total rows',
        'Duplicate rows',
        'Columns',
        'Null values (total)',
        'Benign count',
        'Phishing count',
        'Phishing rate'
    ],
    'Before Cleaning': [
        f'{len(df):,}',
        f'{df.duplicated().sum():,}',
        f'{len(df.columns)}',
        f'{df.isnull().sum().sum():,}',
        f'{label_before[0]:,}',
        f'{label_before[1]:,}',
        f'{label_before[1]/len(df)*100:.2f}%'
    ],
    'After Cleaning': [
        f'{len(df_clean):,}',
        '0',
        f'{len(df_clean.columns)}',
        '0',
        f'{label_after[0]:,}',
        f'{label_after[1]:,}',
        f'{label_after[1]/len(df_clean)*100:.2f}%'
    ]
})

print('── Before / After Comparison ───────────────────')
print(comparison.to_string(index=False))

── Before / After Comparison ───────────────────
             Metric Before Cleaning After Cleaning
         Total rows          15,367         14,823
     Duplicate rows             544              0
            Columns              80             79
Null values (total)           9,669              0
       Benign count           7,781          7,464
     Phishing count           7,586          7,359
      Phishing rate          49.37%         49.65%


In [18]:
# ── Key feature statistics after cleaning ─────────────────────────────────────
key_cols_clean = [c for c in key_cols if c in df_clean.columns]
print('── Key Feature Statistics (After) ──────────────')
df_clean[key_cols_clean].describe().round(4)

── Key Feature Statistics (After) ──────────────


,label,urllen,domain_token_count,entropy_url,symbolcount_url,numberrate_url,numberofdotsinurl
count,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000,14823.0000
mean,0.4965,76.5487,2.5367,0.7204,7.1494,0.0861,2.3642
std,0.5000,33.6261,0.9050,0.0487,3.5324,0.1016,1.7874
min,0.0000,37.0000,2.0000,0.4196,2.0000,0.0000,1.0000
25%,0.0000,54.0000,2.0000,0.6866,5.0000,0.0000,1.0000
50%,0.0000,69.0000,2.0000,0.7221,6.0000,0.0556,2.0000
75%,1.0000,90.0000,3.0000,0.7561,8.0000,0.1250,3.0000
max,1.0000,348.0000,19.0000,0.8697,47.0000,0.7619,20.0000


## 7. Full MongoDB Audit Log

Query all logged operations in order — shows the complete cleaning history.

In [19]:
# Pull all real operations (exclude the template document inserted by mongo_setup.js)
log_docs = list(
    prep_log
    .find({'operation': {'$ne': 'example_template'}})
    .sort('timestamp', 1)
)

log_df = pd.DataFrame(log_docs)[[
    'operation', 'rows_before', 'rows_after', 'rows_affected', 'detail'
]]

print('── MongoDB Preprocessing Log ───────────────────')
print(log_df.to_string(index=False))

── MongoDB Preprocessing Log ───────────────────
            operation  rows_before  rows_after  rows_affected                                                                                                                                                                                                                                                                   detail
  drop_sparse_columns        15367       15367              0                                                                                                                                                                                                                Dropped columns with >40% nulls: ['numberrate_extension']
    remove_duplicates        15367       14823            544                                                                                                                                                                                                                                            

## 8. Export Clean Data

Load clean data into PostgreSQL `clean_urls` table and export to CSV.

In [20]:
# ── Load into PostgreSQL clean_urls ──────────────────────────────────────────
df_clean.to_sql(
    name      = 'clean_urls',
    con       = engine,
    if_exists = 'append',
    index     = False,
    chunksize = 1000
)
print(f'✓ Loaded {len(df_clean):,} rows into clean_urls table')

✓ Loaded 14,823 rows into clean_urls table


In [21]:
# ── Export to CSV ─────────────────────────────────────────────────────────────
os.makedirs('../data/clean', exist_ok=True)
CLEAN_CSV = '../data/clean/clean_urls.csv'
df_clean.to_csv(CLEAN_CSV, index=False)
print(f'✓ Exported to {CLEAN_CSV}')

✓ Exported to ../data/clean/clean_urls.csv


In [22]:
# ── Final verification ────────────────────────────────────────────────────────
with engine.connect() as conn:
    raw_count   = conn.execute(text('SELECT COUNT(*) FROM urls')).scalar()
    clean_count = conn.execute(text('SELECT COUNT(*) FROM clean_urls')).scalar()

csv_check = len(pd.read_csv(CLEAN_CSV))

print('── Final Verification ──────────────────────────')
print(f'  Raw table (urls)       : {raw_count:,} rows')
print(f'  Clean table (clean_urls): {clean_count:,} rows')
print(f'  Exported CSV           : {csv_check:,} rows')
print(f'  Rows removed by cleaning: {raw_count - clean_count:,}')
print()

if clean_count == csv_check:
    print('✓ PostgreSQL and CSV row counts match — preprocessing complete')
else:
    print('✗ Count mismatch between PostgreSQL and CSV — investigate')

── Final Verification ──────────────────────────
  Raw table (urls)       : 15,367 rows
  Clean table (clean_urls): 14,823 rows
  Exported CSV           : 14,823 rows
  Rows removed by cleaning: 544

✓ PostgreSQL and CSV row counts match — preprocessing complete


In [23]:
# ── SQL before/after comparison (from queries.sql Section 5) ─────────────────
sql_compare = '''
    SELECT 'raw'   AS dataset, COUNT(*) AS total_rows,
           ROUND(AVG(urllen), 2) AS avg_urllen,
           ROUND(AVG(entropy_url), 4) AS avg_entropy
    FROM urls
    UNION ALL
    SELECT 'clean' AS dataset, COUNT(*) AS total_rows,
           ROUND(AVG(urllen), 2) AS avg_urllen,
           ROUND(AVG(entropy_url), 4) AS avg_entropy
    FROM clean_urls
'''
print('── SQL Before/After Comparison ─────────────────')
display(pd.read_sql(sql_compare, engine))

── SQL Before/After Comparison ─────────────────


,dataset,total_rows,avg_urllen,avg_entropy
0,clean,14823,76.55,0.7204
1,raw,15367,76.04,0.7217


---
## Summary

| Operation | Rows Affected | Detail |
|---|---|---|
| Drop sparse columns | 0 rows | `numberrate_extension` dropped (>40% null) |
| Remove duplicates | *544* | Exact duplicate rows removed |
| Median imputation | 0 rows | Nulls filled with column medians |
| Dtype verification | 0 rows | All columns confirmed numeric |
| Label integrity | 0 rows | All labels confirmed 0 or 1 |
| **Total rows removed** | *544* | Raw: 15,367 → Clean: *14823* |

**Next:** `03_visualization.ipynb` — EDA and charts using the clean dataset.